# PrismaCoder - Evaluation Notebook (PrismaBench / Spider / BIRD)

Runs the full evaluation flow: benchmark build or load -> model inference over tasks -> gate-based scoring -> per-category/per-level reports with Wilson CIs -> error analysis -> charts -> leaderboard. Static mode works without Node; pass `--full-gates` equivalents when Node >= 18.18 is present.


In [ ]:
# Cell 1 - setup
import os, subprocess, sys
REPO_DIR = '/content/PrismaCoder'
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/prismacoder/PrismaCoder.git', REPO_DIR], check=True)
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
%pip install -q matplotlib pandas

MODEL_ID = 'prismacoder/PrismaCoder-3B'
SEED = 20260830
BENCH_SCALE = 0.05        # 0.05 -> 60-task smoke; 1.0 -> full 1,200
TASKS_LIMIT = 30          # cap inference cost in the notebook
FULL_GATES = False


In [ ]:
# Cell 2 - build (or load) PrismaBench deterministically
from benchmark.generator.generate_benchmark import BenchmarkBuilder
from prismacoder.io_utils import write_jsonl, read_jsonl
from pathlib import Path

builder = BenchmarkBuilder(SEED, BENCH_SCALE)
public_rows, private_rows, provenance = builder.build()
Path('benchmark/public').mkdir(parents=True, exist_ok=True)
write_jsonl('benchmark/public/prismabench_public.jsonl', public_rows)
Path('benchmark/private').mkdir(parents=True, exist_ok=True)
write_jsonl('benchmark/private/private_keys.jsonl', private_rows)
print(f'benchmark: {len(public_rows)} public + {len(private_rows)} private tasks')


In [ ]:
# Cell 3 - run the model over the tasks (static-gate scoring harness)
from prismacoder.inference.pipeline import PrismaCoderPipeline
from benchmark.evaluator.evaluate import score_task, load_reference_index
from prismacoder.eval_metrics.registry import default_registry, aggregate_results
from prismacoder.verification.verifier import LightValidator

pipe = PrismaCoderPipeline.from_local(MODEL_ID, temperature=0.0, repair=False)
light = LightValidator()
registry = default_registry()
refs = load_reference_index(Path('benchmark/private/private_keys.jsonl'), public_rows)

rows = []
for task in public_rows[:TASKS_LIMIT]:
    artifact = pipe.extract_code(pipe.raw_generate(
        pipe._build_messages(task['instruction'], task['context']['schema'])))
    submission = {'bench_id': task['bench_id'], 'output': artifact,
                  'latency_ms': pipe.last_stats.latency_ms, 'memory_mb': 0}
    row = score_task(task, submission, refs.get(task['bench_id']), None, light, registry)
    rows.append(row)
    print('.', end='', flush=True)
print()
aggregate_results(rows)


In [ ]:
# Cell 4 - write results + leaderboard entry
import time
from prismacoder.io_utils import write_json
results = {'model': MODEL_ID,
           'evaluated_at': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
           'mode': 'static' if not FULL_GATES else 'full_gates',
           'aggregate': aggregate_results(rows), 'rows': rows}
os.makedirs('benchmark/leaderboard', exist_ok=True)
write_json('benchmark/leaderboard/notebook_results.json', results)
subprocess.run([sys.executable, 'benchmark/evaluator/leaderboard.py'], check=True)
print(open('benchmark/leaderboard/LEADERBOARD.md').read())


In [ ]:
# Cell 5 - error analysis (E1-E10 Pareto of fixes)
subprocess.run([sys.executable, 'evaluation/error_analysis.py',
                '--results', 'benchmark/leaderboard/notebook_results.json',
                '--out-md', 'reports/error_analysis.md',
                '--out-json', 'reports/error_analysis.json'], check=True)
print(open('reports/error_analysis.md').read())


In [ ]:
# Cell 6 - charts: category comparison, difficulty ladder, gate profile, taxonomy
subprocess.run([sys.executable, 'evaluation/plots.py',
                f'{MODEL_ID}=benchmark/leaderboard/notebook_results.json',
                '--out-dir', 'reports/figures'], check=True)
from IPython.display import Image, display  # type: ignore
for fig in sorted(os.listdir('reports/figures')):
    print(fig)


In [ ]:
# Cell 7 - (optional) cross-benchmark predictions for Spider/BIRD
# Requires the public dev sets downloaded (docs/REPRODUCIBILITY.md) and a GPU session.
run_cross = False
if run_cross:
    subprocess.run([sys.executable, 'evaluation/spider_adapter.py',
                    '--benchmark', 'spider', '--data', 'data/spider/dev.json',
                    '--databases', 'data/spider/database', '--model', MODEL_ID,
                    '--limit', '100', '--out', 'reports/spider_predictions.json'], check=True)


## Reading the results

- **pass@1** with Wilson 95% CI is the primary metric; per-category and per-level breakdowns localize weaknesses; the gate profile shows *where* tasks die (V1 validate vs V3 tsc vs V5 vitest).
- The **E1-E10** taxonomy (report Table 11.3) turns failures into a fix backlog: see `reports/error_analysis.md` Pareto section.
- For paper-grade numbers use the full 1,200-task set with `--full-gates` (Node + docker verifier) and submit via `benchmark/evaluator/`.
